# Web Search with LangChain Agents

This reference notebook compares an agent that has no web access with an agent that can call Tavily search as a tool.

## Learning goals

- Understand why a language model's built-in knowledge may be incomplete for current events.
- Define a Python function as a LangChain tool using the `@tool` decorator.
- Test the search tool directly before giving it to an agent.
- Let an agent decide when a web search is needed.

## Before you run the notebook

1. Add your model-provider API key to the environment or `.env` file.
2. Add `TAVILY_API_KEY` as well; TavilyClient reads it from the environment.
3. Run the cells from top to bottom. The final question is time-sensitive, so its answer will change as the web changes.

The first section is a baseline. The second section adds live retrieval so the agent can ground its answer in current search results.

## 1. Baseline: No Web Search

This agent can answer from its model knowledge only. It has no tool that can retrieve information published after its training data or verify a live fact.

In [12]:
# Load API keys and other local settings from the project's .env file.
from dotenv import load_dotenv

load_dotenv()

True

In [13]:
from langchain.agents import create_agent

# This baseline agent has no tools, so it cannot perform live web retrieval.
agent = create_agent(model="gpt-5-nano")

In [14]:
from langchain.messages import HumanMessage

# This question asks about the model's knowledge boundary, not a live fact.
question = HumanMessage(content="How up to date is your training knowledge?")
response = agent.invoke({"messages": [question]})

In [15]:
# The final message contains the assistant's answer to the question.
print(response["messages"][-1].content)

My training data goes up to June 2024, so I don’t have first-hand knowledge of events or developments that happened after that date. I don’t have real-time web access by default, so I can’t pull in fresh information unless your platform provides a browsing tool or you share sources with me.

If you need up-to-date info, you can:
- Paste articles or data here and I’ll summarize or analyze them.
- Tell me the topic and I’ll outline the latest known patterns up to 2024 and suggest how to verify current details with reputable sources.


### Baseline summary

Without a search tool, the agent cannot verify current information. This is useful as a control example: adding a tool changes the agent's capabilities, not the model's underlying training data.

## 2. Add a Web Search Tool

A LangChain tool is a callable function with a clear name, description, and input schema. The agent can use that schema to decide when to call the function.

In [16]:
from typing import Any

from langchain.tools import tool
from tavily import TavilyClient

# TavilyClient reads TAVILY_API_KEY from the environment loaded above.
tavily_client = TavilyClient()


@tool
def web_search(query: str) -> dict[str, Any]:
    """Search the web for information relevant to a user's question."""
    # Keeping the input as a string gives the agent one simple argument to fill.
    return tavily_client.search(query)

# Test the tool directly before giving it to the agent.
web_search.invoke("Who is the current mayor of San Francisco?")

{'query': 'Who is the current mayor of San Francisco?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://ballotpedia.org/Daniel_Lurie',
   'title': 'Daniel Lurie',
   'content': "#### Sign up to receive Ballotpedia's daily newsletter\n\nEmail \\\n\nFirst Name\n\nPlease complete the Captcha above\n\n#### Ballotpedia on Facebook\n\nShare this page\n\nFollow Ballotpedia\n\n#### Ballotpedia on Twitter\n\nShare this page\n\nFollow Ballotpedia\n\nBallotpedia Logo\nBallotpedia Logo\n\n# Daniel Lurie\n\nSilhouette Placeholder Image.png\n\nReport an officeholder change\n\nDaniel Lurie is the mayor of San Francisco, California. He assumed office on January 8, 2025. His current term ends on January 8, 2029. [...] ## Footnotes\n\n| Political offices | | |\n --- \n| Preceded by   London Breed | Mayor of San Francisco   2025-Present | Succeeded by   - |\n\n|  |  |  |  |  |  |  |  |  |  |\n ---  ---  ---  ---  --- |\n| | v • e 2024 Municipal Elections | | | 

In [17]:
# Give the search tool to a new agent; the baseline agent remains tool-free.
agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search],
)

question = HumanMessage(content="Who is the current mayor of San Francisco?")

# The agent can decide to call web_search before producing its answer.
response = agent.invoke({"messages": [question]})

In [18]:
from pprint import pprint

# Inspect every message to see the tool call and the final assistant response.
pprint(response["messages"])

[HumanMessage(content='Who is the current mayor of San Francisco?', additional_kwargs={}, response_metadata={}, id='a6f289e3-1a8f-494c-abc4-c7a73857775a'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 356, 'prompt_tokens': 138, 'total_tokens': 494, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EPofgmb6asWrH3OiyGa61CkX0VCk3', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0b9af-97c9-7c62-9c67-c8b2b18e4aea-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'current mayor of San Francisco 2024 2025 2026'}, 'id': 'call_LhYjW8EfNLSbnoFuJTdGIlvp', 'type': 'tool_call'}], invali

In [19]:
print(response["messages"][-1].content)

Daniel Lurie. He took office on January 8, 2025, as the 46th mayor of San Francisco, succeeding London Breed. Would you like a quick bio or current priorities?


### Web-search summary

The direct invocation checks that Tavily is reachable and returns search data. The agent invocation then demonstrates tool selection: the model receives the tool schema, calls `web_search` when appropriate, and incorporates the returned results into its answer. Search results should still be checked for source quality and recency.

## Conclusion and reference checklist

Use the no-tool agent for questions that do not require current information. Use the search-enabled agent for time-sensitive or externally verifiable questions.

Reusable pattern:

1. Load credentials before constructing the client.
2. Define a typed tool with a concise docstring.
3. Test the tool directly so integration errors are easier to isolate.
4. Pass the tool to the agent with `tools=[web_search]`.
5. Inspect the returned messages to verify that the tool was called and that the final answer reflects the retrieved results.

Live search improves freshness, but it does not guarantee correctness. In production, handle API failures, inspect sources, set search limits, and make the application's citation or verification policy explicit.